##Inserting datasets

In [0]:
trips = spark.read.parquet(
    "/Volumes/workspace/default/trip/yellow_tripdata_2024-01.parquet"
)

zones = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "/Volumes/workspace/default/trip/taxi_zone_lookup.csv"
)

##EDA(Exploratory Data Analysis)

checking schema

In [0]:
trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [0]:
zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



taking overview of data

In [0]:
trips.show(5)
zones.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

rows counts

In [0]:
print("Trips:", trips.count())
print("Zones:", zones.count())

Trips: 2964624
Zones: 265


droping duplicates

In [0]:
trips_clean = trips.dropDuplicates()
zones_clean = zones.dropDuplicates()

In [0]:
print("Trips after duplicate removal:", trips_clean.count())
if (trips_clean.count() - trips.count()) == 0:
  print("NO duplicates in trips")
print()
print("Zones after duplicate removal:", zones_clean.count())
if (zones_clean.count() - zones.count()) == 0:
  print("NO duplicates in zones")

Trips after duplicate removal: 2964624
NO duplicates in trips

Zones after duplicate removal: 265
NO duplicates in zones


##NULL Profiling

NULL count in each column

In [0]:
from pyspark.sql import functions as F

trips_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in trips_clean.columns
]).show()

zones_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in zones_clean.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         140162|            0|    140162|            140162|           0|           0|           0|          0|    0|      0|         

In [0]:
trips_clean.filter(
    F.col("passenger_count").isNull()
).select(
    "VendorID",
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
).show(20)

+--------+---------------+----------+------------------+--------------------+-----------+
|VendorID|passenger_count|RatecodeID|store_and_fwd_flag|congestion_surcharge|Airport_fee|
+--------+---------------+----------+------------------+--------------------+-----------+
|       2|           NULL|      NULL|              NULL|                NULL|       NULL|
|       1|           NULL|      NULL|              NULL|                NULL|       NULL|
|       2|           NULL|      NULL|              NULL|                NULL|       NULL|
|       2|           NULL|      NULL|              NULL|                NULL|       NULL|
|       2|           NULL|      NULL|              NULL|                NULL|       NULL|
|       2|           NULL|      NULL|              NULL|                NULL|       NULL|
|       2|           NULL|      NULL|              NULL|                NULL|       NULL|
|       1|           NULL|      NULL|              NULL|                NULL|       NULL|
|       1|

In [0]:
trips_clean.filter(
    F.col("passenger_count").isNull()
).select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "total_amount"
).show(20)

+--------+--------------------+---------------------+------------+------------+------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|payment_type|fare_amount|total_amount|
+--------+--------------------+---------------------+------------+------------+------------+-----------+------------+
|       2| 2024-01-01 00:50:28|  2024-01-01 01:17:12|         236|         164|           0|      29.17|       33.17|
|       1| 2024-01-01 00:48:14|  2024-01-01 01:31:08|         113|          41|           0|      36.02|       40.02|
|       2| 2024-01-01 00:57:45|  2024-01-01 01:16:44|         263|         116|           0|      24.19|       28.19|
|       2| 2024-01-01 00:54:09|  2024-01-01 01:20:04|         113|         141|           0|      25.15|       29.15|
|       2| 2024-01-01 00:49:22|  2024-01-01 01:03:49|         209|         164|           0|      21.76|       30.91|
|       2| 2024-01-01 00:54:58|  2024-01-01 01:14:37|   

In [0]:
trips_clean.filter(
    F.col("passenger_count").isNull()
).select(
    "payment_type"
).groupBy("payment_type").count().show()

+------------+------+
|payment_type| count|
+------------+------+
|           0|140162|
+------------+------+



In [0]:
trips_clean.filter(
    F.col("payment_type") == 0
).select(
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
).show(20)

+---------------+----------+------------------+--------------------+-----------+
|passenger_count|RatecodeID|store_and_fwd_flag|congestion_surcharge|Airport_fee|
+---------------+----------+------------------+--------------------+-----------+
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|              NULL|                NULL|       NULL|
|           NULL|      NULL|

In [0]:
trips_clean.filter(
    F.col("payment_type") == 0
).count()

140162

Rows with `payment_type = 0` contain NULL values in `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee`.

### NULL Value Analysis

Rows with `payment_type = 0` contain NULL values in:
- `passenger_count`
- `RatecodeID`
- `store_and_fwd_flag`
- `congestion_surcharge`
- `Airport_fee`

These records were retained because the intended values for the NULL fields
could not be reliably inferred from the available data.

#checking for invalid data

for passenger count roow

In [0]:
trips_clean.filter(
    F.col("passenger_count") <= 0
).groupBy(
    "passenger_count"
).count().show()

+---------------+-----+
|passenger_count|count|
+---------------+-----+
|              0|31465|
+---------------+-----+



In [0]:
trips_clean.filter(
    F.col("passenger_count") == 0
).groupBy(
    "payment_type"
).count().show()

+------------+-----+
|payment_type|count|
+------------+-----+
|           1|25040|
|           2| 5229|
|           4|  223|
|           3|  973|
+------------+-----+



### Passenger Count Validation

A total of **31,465 records** have `passenger_count = 0`.

These records are **not associated with `payment_type = 0`**.  
Their payment types are:

- `payment_type = 1`: 25,040 records
- `payment_type = 2`: 5,229 records
- `payment_type = 3`: 973 records
- `payment_type = 4`: 223 records

Therefore, `passenger_count = 0` is treated as a **separate data-quality anomaly** and will be investigated before deciding whether these records should be removed or corrected.

In [0]:
trips_clean.filter(
    F.col("passenger_count") == 0
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type"
).show(20)

+--------------------+---------------------+-------------+-----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|payment_type|
+--------------------+---------------------+-------------+-----------+------------+------------+
| 2024-01-01 00:16:48|  2024-01-01 00:18:59|          0.4|        5.1|        12.6|           1|
| 2024-01-01 00:30:40|  2024-01-01 00:58:40|          3.0|       25.4|        30.4|           2|
| 2024-01-01 00:28:09|  2024-01-01 00:55:11|          3.1|       26.1|       38.85|           1|
| 2024-01-01 00:29:31|  2024-01-01 00:41:08|          1.7|       11.4|        21.3|           1|
| 2024-01-01 00:40:40|  2024-01-01 01:00:12|          2.5|       18.4|       28.05|           1|
| 2024-01-01 00:54:36|  2024-01-01 01:23:38|          3.8|       26.8|       38.15|           1|
| 2024-01-01 00:34:01|  2024-01-01 00:50:29|          2.2|       16.3|       25.55|           1|
| 2024-01-01 00:50:13|  2024-0

### Passenger Count Anomaly

A total of **31,465 records** have `passenger_count = 0`.

These records contain valid trip distances, pickup/dropoff times, and fare amounts, and are distributed across normal payment types. No clear pattern was found to determine the actual passenger count.

Therefore, these `0` values will be treated as **unknown passenger counts** and converted to `NULL` rather than assigning an arbitrary value.

In [0]:
trips_clean = trips_clean.withColumn(
    "passenger_count",
    F.when(
        F.col("passenger_count") == 0,
        None
    ).otherwise(F.col("passenger_count"))
)

In [0]:
trips_clean.groupBy("passenger_count").count().orderBy("passenger_count").show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL| 171627|
|              1|2188739|
|              2| 405103|
|              3|  91262|
|              4|  51974|
|              5|  33506|
|              6|  22353|
|              7|      8|
|              8|     51|
|              9|      1|
+---------------+-------+



there are 9 passengers in one trip...checking if its valid 

In [0]:
trips_clean.filter(
    F.col("passenger_count") == 9
).select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount",
    "payment_type"
).show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|total_amount|payment_type|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+------------+
|1       |2024-01-17 07:17:21 |2024-01-17 07:26:21  |9              |1.8          |90          |161         |11.4       |18.45       |1           |
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+------------+



data looks normal , so keeping value 9 for passengers 

. now checking trip distance

In [0]:
trips_clean.filter(
    F.col("trip_distance") <= 0
).groupBy(
    "trip_distance"
).count().show()

+-------------+-----+
|trip_distance|count|
+-------------+-----+
|          0.0|60371|
+-------------+-----+



In [0]:
trips_clean.filter(
    F.col("trip_distance") == 0
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount",
    "payment_type"
).show(20)

+--------------------+---------------------+---------------+------------+------------+-----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|PULocationID|DOLocationID|fare_amount|total_amount|payment_type|
+--------------------+---------------------+---------------+------------+------------+-----------+------------+------------+
| 2024-01-01 00:35:57|  2024-01-01 00:36:07|              1|          42|          42|        3.0|         5.5|           2|
| 2024-01-01 00:31:10|  2024-01-01 00:36:41|              2|         264|         162|        6.5|        11.5|           2|
| 2024-01-01 00:14:26|  2024-01-01 00:14:35|              1|          79|          79|       15.3|       22.56|           1|
| 2024-01-01 00:33:28|  2024-01-01 00:33:57|              2|         162|         162|       -3.0|        -8.0|           4|
| 2024-01-01 00:56:55|  2024-01-01 00:57:03|              1|          79|          79|       36.0|        49.5|           1|


### Trip Distance Anomaly

A total of **60,371 records** have `trip_distance = 0`.

After inspecting sample records, these trips contain valid timestamps, locations, passenger counts, and fare information. Therefore, zero-distance trips are treated as valid edge cases and retained in the dataset.

now we check for fare amount

In [0]:
trips_clean.filter(
    F.col("fare_amount") < 0
).groupBy(
    "fare_amount"
).count().show()

+-----------+-----+
|fare_amount|count|
+-----------+-----+
|      -40.1|   95|
|      -70.0| 2023|
|       -4.4|  938|
|       -3.0| 3993|
|       -3.7|  917|
|       -6.5| 1506|
|       -7.2| 1479|
|      -51.3|   40|
|      -22.7|    2|
|       -9.3| 1235|
|       -8.6| 1373|
|      -66.0|   18|
|      -19.8|  294|
|      -11.4| 1055|
|      -10.0| 1277|
|      -14.9|  658|
|      -23.3|  195|
|      -12.8|  911|
|      -27.5|  170|
|       -5.1| 1180|
+-----------+-----+
only showing top 20 rows


In [0]:
trips_clean.filter(
    F.col("fare_amount") < 0
).select(
    "fare_amount",
    "total_amount",
    "payment_type",
    "trip_distance","passenger_count",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).show(20)

+-----------+------------+------------+-------------+---------------+--------------------+---------------------+
|fare_amount|total_amount|payment_type|trip_distance|passenger_count|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+------------+------------+-------------+---------------+--------------------+---------------------+
|      -40.1|       -45.1|           4|          9.6|              1| 2024-01-01 00:30:18|  2024-01-01 00:57:39|
|      -70.0|       -74.0|           4|         0.01|              1| 2024-01-01 00:17:55|  2024-01-01 00:18:05|
|       -4.4|        -9.4|           4|         0.42|              1| 2024-01-01 00:09:37|  2024-01-01 00:11:56|
|      -13.5|       -18.5|           4|         1.78|              3| 2024-01-01 00:32:09|  2024-01-01 00:45:40|
|      -13.5|       -18.5|           4|         2.16|              1| 2024-01-01 00:18:24|  2024-01-01 00:30:39|
|      -33.1|       -38.1|           2|         5.48|              1| 2024-01-01 00:42:02|  2024

In [0]:
trips_clean.filter(
    F.col("fare_amount") < 0
).select(
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee"
).show(20)

+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|      -40.1| -1.0|   -0.5|       0.0|         0.0|                 -1.0|       -45.1|                -2.5|        0.0|
|      -70.0|  0.0|   -0.5|       0.0|         0.0|                 -1.0|       -74.0|                -2.5|        0.0|
|       -4.4| -1.0|   -0.5|       0.0|         0.0|                 -1.0|        -9.4|                -2.5|        0.0|
|      -13.5| -1.0|   -0.5|       0.0|         0.0|                 -1.0|       -18.5|                -2.5|        0.0|
|      -13.5| -1.0|   -0.5|       0.0|         0.0|                 -1.0|       -18.5|                -2.5|        0.0|
|      -33.1| -1.0|   -0.5|       0.0|  

### Negative Fare Records

Some records contain negative values across multiple monetary fields,
including `fare_amount`, `taxes`, `surcharges`, and `total_amount`.
The negative values are internally consistent across the related monetary
columns, so they were retained rather than arbitrarily replaced.

now we check total amount

In [0]:
trips_clean.filter(
    (F.col("fare_amount") >= 0) &
    (F.col("total_amount") <= 0)
).groupBy(
    "total_amount"
).count().orderBy(
    "total_amount"
).show()

+------------+-----+
|total_amount|count|
+------------+-----+
|       -5.75|    6|
|        -5.0|    1|
|        -4.0|   77|
|        -3.5|    2|
|       -3.25|   11|
|       -2.75|    5|
|        -1.5|   11|
|        -1.0|    7|
|         0.0|  416|
+------------+-----+



In [0]:
trips_clean.filter(
    (F.col("fare_amount") >= 0) &
    (F.col("total_amount") <= 0)
).select(
    "fare_amount",
    "total_amount",
    "payment_type",
    "trip_distance",
    "passenger_count",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).show(20)

+-----------+------------+------------+-------------+---------------+--------------------+---------------------+
|fare_amount|total_amount|payment_type|trip_distance|passenger_count|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+------------+------------+-------------+---------------+--------------------+---------------------+
|        0.0|         0.0|           3|          0.0|              1| 2024-01-02 12:39:06|  2024-01-02 12:39:41|
|        0.0|         0.0|           1|          0.0|              1| 2024-01-01 13:46:41|  2024-01-01 14:12:06|
|        0.0|         0.0|           2|          0.0|              1| 2024-01-02 08:11:15|  2024-01-02 08:12:10|
|        0.0|         0.0|           4|          8.1|              1| 2024-01-02 14:12:16|  2024-01-02 14:34:18|
|        0.0|         0.0|           2|          0.0|              1| 2024-01-01 13:03:22|  2024-01-01 13:03:24|
|        0.0|         0.0|           4|          0.0|              1| 2024-01-01 21:33:28|  2024

In [0]:
trips_clean.filter(
    (F.col("fare_amount") >= 0) &
    (F.col("total_amount") <= 0)
).groupBy(
    "fare_amount",
    "total_amount"
).count().orderBy(
    "fare_amount",
    "total_amount"
).show(50)

+-----------+------------+-----+
|fare_amount|total_amount|count|
+-----------+------------+-----+
|        0.0|       -5.75|    6|
|        0.0|        -5.0|    1|
|        0.0|        -4.0|   77|
|        0.0|        -3.5|    2|
|        0.0|       -3.25|   11|
|        0.0|       -2.75|    5|
|        0.0|        -1.5|   11|
|        0.0|        -1.0|    7|
|        0.0|         0.0|  416|
+-----------+------------+-----+



### Non-positive Total Amount

A total of **520 records** have `total_amount <= 0`.
Among these, **416 records have total_amount = 0**, while **104 records
have negative total_amount values**.

These records show different trip characteristics and no reliable pattern
was found to infer the intended monetary values. Therefore, the original
values were retained rather than applying arbitrary replacements.

.now ratecode id

In [0]:
trips_clean.filter(
    F.col("RatecodeID").isNotNull()
).groupBy(
    "RatecodeID"
).count().orderBy(
    "RatecodeID"
).show()

+----------+-------+
|RatecodeID|  count|
+----------+-------+
|         1|2663350|
|         2|  98713|
|         3|   7954|
|         4|   6365|
|         5|  19410|
|         6|      7|
|        99|  28663|
+----------+-------+



In [0]:
trips_clean.filter(
    F.col("RatecodeID") == 99
).select(
    "RatecodeID",
    "payment_type",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "PULocationID",
    "DOLocationID"
).show(20)

+----------+------------+-------------+-----------+------------+------------+------------+
|RatecodeID|payment_type|trip_distance|fare_amount|total_amount|PULocationID|DOLocationID|
+----------+------------+-------------+-----------+------------+------------+------------+
|        99|           1|          0.4|       15.5|        17.0|         242|         185|
|        99|           1|          1.9|       18.5|        20.0|         222|          39|
|        99|           1|         11.8|       44.5|        46.0|         138|         181|
|        99|           1|          6.0|       27.5|        29.0|          91|         228|
|        99|           1|          1.7|       17.5|        19.0|         178|         123|
|        99|           1|         13.9|       48.5|       56.94|         215|         151|
|        99|           1|         12.7|       40.5|       48.94|         225|         152|
|        99|           1|          0.0|       22.5|        24.0|          83|         197|

In [0]:
trips_clean.filter(
    F.col("RatecodeID") == 99
).groupBy(
    "payment_type"
).count().orderBy(
    "payment_type"
).show()

+------------+-----+
|payment_type|count|
+------------+-----+
|           1|28649|
|           2|    5|
|           3|    9|
+------------+-----+



In [0]:
trips_clean.filter(
    F.col("payment_type") == 1
).groupBy(
    "RatecodeID"
).count().orderBy(
    "RatecodeID"
).show()

+----------+-------+
|RatecodeID|  count|
+----------+-------+
|         1|2184991|
|         2|  79244|
|         3|   5744|
|         4|   4470|
|         5|  15948|
|        99|  28649|
+----------+-------+



### RatecodeID Anomaly

`RatecodeID = 99` occurs in **28,663 records**. These records were inspected
and showed plausible trip distance, fare, total amount, and location values.
Although `payment_type = 1` is overwhelmingly dominant among these records,
no sufficient evidence was found to classify `99` as invalid.

Therefore, `RatecodeID = 99` was retained without modification.

now store_and_fwd_flag

In [0]:
trips_clean.filter(
    F.col("store_and_fwd_flag").isNotNull()
).groupBy(
    "store_and_fwd_flag"
).count().orderBy(
    "store_and_fwd_flag"
).show()

+------------------+-------+
|store_and_fwd_flag|  count|
+------------------+-------+
|                 N|2813126|
|                 Y|  11336|
+------------------+-------+



### store_and_fwd_flag

The `store_and_fwd_flag` column contains two non-null values: `N` and `Y`.
Both values are valid categorical values, while **140,162 records contain NULL**.

Since the NULL values cannot be reliably inferred as either `N` or `Y`,
they were retained as NULL rather than assigning an arbitrary value.

now pick up location id

In [0]:
trips_clean.filter(
    F.col("PULocationID").isNull() |
    (F.col("PULocationID") <= 0)
).groupBy(
    "PULocationID"
).count().show()

+------------+-----+
|PULocationID|count|
+------------+-----+
+------------+-----+



In [0]:
trips_clean.join(
    zones_clean,
    trips_clean.PULocationID == zones_clean.LocationID,
    "left_anti"
).select(
    "PULocationID"
).groupBy(
    "PULocationID"
).count().show()

+------------+-----+
|PULocationID|count|
+------------+-----+
+------------+-----+



now drop off location id

In [0]:
trips_clean.filter(
    F.col("DOLocationID").isNull() |
    (F.col("DOLocationID") <= 0)
).groupBy(
    "DOLocationID"
).count().show()

+------------+-----+
|DOLocationID|count|
+------------+-----+
+------------+-----+



In [0]:
trips_clean.join(
    zones_clean,
    trips_clean.DOLocationID == zones_clean.LocationID,
    "left_anti"
).select(
    "DOLocationID"
).groupBy(
    "DOLocationID"
).count().show()

+------------+-----+
|DOLocationID|count|
+------------+-----+
+------------+-----+



### Location ID Validation

`PULocationID` and `DOLocationID` represent the pickup and drop-off taxi zones.
Both columns were checked for NULL and non-positive values, and no invalid
values were found.

Both location IDs were also validated against `zones.LocationID`.
No unmatched location IDs were found.

Therefore, `PULocationID` and `DOLocationID` were retained without modification.

now payment_type

In [0]:
trips_clean.groupBy(
    "payment_type"
).count().orderBy(
    "payment_type"
).show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|           0| 140162|
|           1|2319046|
|           2| 439191|
|           3|  19597|
|           4|  46628|
+------------+-------+



### Payment Type Validation

The `payment_type` column contains values `0`, `1`, `2`, `3`, and `4`.

The `payment_type = 0` records were previously investigated and were found
to correspond to the same 140,162 records containing NULL values in several
related columns. These records were retained without modification.

The remaining payment type values were observed with substantial record
counts and no invalid or unexpected values were identified.

Therefore, `payment_type` was retained without modification.

now , 'extra' column

In [0]:
trips_clean.filter(
    F.col("extra") < 0
).groupBy(
    "extra"
).count().orderBy(
    "extra"
).show()

+-----+-----+
|extra|count|
+-----+-----+
| -7.5|  227|
| -6.0|  319|
| -5.0| 1146|
| -3.5|    1|
| -2.5| 5564|
| -1.5|    3|
| -1.0|10287|
|-0.04|    1|
+-----+-----+



### Extra Validation

The `extra` column contains several negative values, including `-7.5`, `-6.0`,
`-5.0`, `-2.5`, and `-1.0`. These values were inspected and were not
classified as invalid based on the available data.

Therefore, the original `extra` values were retained without modification.

now mta_tax

In [0]:
trips_clean.filter(
    F.col("mta_tax") < 0
).groupBy(
    "mta_tax"
).count().orderBy(
    "mta_tax"
).show()

+-------+-----+
|mta_tax|count|
+-------+-----+
|   -0.5|34434|
+-------+-----+



In [0]:
trips_clean.filter(
    F.col("mta_tax") < 0
).select(
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "payment_type"
).show(20)

+-----------+-----+-------+----------+------------+---------------------+------------+------------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|
+-----------+-----+-------+----------+------------+---------------------+------------+------------+
|      -40.1| -1.0|   -0.5|       0.0|         0.0|                 -1.0|       -45.1|           4|
|      -70.0|  0.0|   -0.5|       0.0|         0.0|                 -1.0|       -74.0|           4|
|       -4.4| -1.0|   -0.5|       0.0|         0.0|                 -1.0|        -9.4|           4|
|      -13.5| -1.0|   -0.5|       0.0|         0.0|                 -1.0|       -18.5|           4|
|      -13.5| -1.0|   -0.5|       0.0|         0.0|                 -1.0|       -18.5|           4|
|      -33.1| -1.0|   -0.5|       0.0|         0.0|                 -1.0|       -38.1|           2|
|       -3.0| -1.0|   -0.5|       0.0|         0.0|                 -1.0|        -8.0|           4|


### MTA Tax Validation

The `mta_tax` column contains **34,434 records with a value of -0.5**.

These records were investigated and the negative `mta_tax` values were found
alongside other negative monetary values such as `fare_amount`,
`extra`, `improvement_surcharge`, and `total_amount`.

Therefore, the negative `mta_tax` values were not treated as isolated invalid
values and were retained without modification.

now tip amount

In [0]:
trips_clean.filter(
    F.col("tip_amount") < 0
).groupBy(
    "tip_amount"
).count().orderBy(
    "tip_amount"
).show()

+----------+-----+
|tip_amount|count|
+----------+-----+
|     -80.0|    1|
|    -66.02|    1|
|     -65.1|    1|
|     -52.0|    1|
|    -37.58|    1|
|     -33.0|    1|
|    -22.24|    1|
|     -22.0|    2|
|    -17.59|    1|
|    -16.19|    3|
|    -14.77|    1|
|     -14.0|    1|
|    -13.65|    1|
|     -11.8|    1|
|     -10.0|    2|
|     -8.18|    1|
|     -7.05|    1|
|     -6.65|    1|
|     -5.04|    1|
|      -5.0|    1|
+----------+-----+
only showing top 20 rows


In [0]:
trips_clean.filter(
    F.col("tip_amount") < 0
).select(
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "payment_type"
).show(30)

+-----------+-----+-------+----------+------------+---------------------+------------+------------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|
+-----------+-----+-------+----------+------------+---------------------+------------+------------+
|       -3.0| -2.5|   -0.5|     -0.01|         0.0|                 -1.0|       -7.01|           3|
|       -3.0| -2.5|   -0.5|     -0.01|         0.0|                 -1.0|       -7.01|           3|
|      -35.9| -1.0|   -0.5|     -8.18|         0.0|                 -1.0|      -49.08|           1|
|       -3.0| -1.0|   -0.5|     -0.01|         0.0|                 -1.0|       -5.51|           3|
|       -3.0| -2.5|   -0.5|     -0.01|         0.0|                 -1.0|       -7.01|           3|
|      -19.1| -6.0|   -0.5|     -6.65|         0.0|                 -1.0|       -35.0|           1|
|       -3.0|  0.0|   -0.5|     -0.01|         0.0|                 -1.0|       -4.51|           3|


### Tip Amount Validation

Some records contain negative values in `tip_amount`. These records were
inspected along with the other monetary columns and were found to generally
occur as part of broader negative-value transactions, where `fare_amount`
and `total_amount` are also negative.

Therefore, the negative `tip_amount` values were retained without
modification.

now tolls amount

In [0]:
trips_clean.filter(
    F.col("tolls_amount") < 0
).groupBy(
    "tolls_amount"
).count().orderBy(
    "tolls_amount"
).show()

+------------+-----+
|tolls_amount|count|
+------------+-----+
|       -80.0|    1|
|       -60.0|    1|
|      -56.64|    1|
|      -55.34|    1|
|      -54.02|    1|
|      -52.57|    1|
|       -50.0|    2|
|      -49.26|    1|
|      -48.75|    1|
|      -47.26|    1|
|       -45.0|    1|
|      -42.75|    1|
|      -42.32|    1|
|      -42.22|    1|
|      -42.13|    1|
|       -40.0|    2|
|      -39.38|    1|
|      -38.02|    1|
|      -37.06|    1|
|      -35.48|    1|
+------------+-----+
only showing top 20 rows


### Tolls Amount Validation

Some records contain negative values in `tolls_amount`. These records were
found to occur as part of broader negative-value transactions, where other
monetary fields such as `fare_amount` and `total_amount` are also negative.

Therefore, the negative `tolls_amount` values were retained without
modification.

now improvement surcharges

In [0]:
trips_clean.filter(
    F.col("improvement_surcharge") < 0
).groupBy(
    "improvement_surcharge"
).count().orderBy(
    "improvement_surcharge"
).show()

+---------------------+-----+
|improvement_surcharge|count|
+---------------------+-----+
|                 -1.0|35500|
|                 -0.3|    2|
+---------------------+-----+



### Improvement Surcharge Validation

The `improvement_surcharge` column contains negative values, mainly `-1.0`
(35,500 records) and `-0.3` (2 records).

The negative values were observed as part of broader negative-value
transactions involving other monetary fields. Therefore, they were not
classified as invalid based only on their sign.

The original `improvement_surcharge` values were retained without
modification.

now  congestion_surcharge

In [0]:
trips_clean.filter(
    F.col("congestion_surcharge") < 0
).groupBy(
    "congestion_surcharge"
).count().orderBy(
    "congestion_surcharge"
).show()

### Congestion Surcharge Validation

The `congestion_surcharge` column contains negative values, mainly `-2.5`
(28,824 records) and `-0.75` (1 record).

These negative values were observed as part of broader negative-value
transactions involving other monetary fields. Therefore, they were not
classified as invalid based only on their sign.

The original `congestion_surcharge` values were retained without
modification. NULL values were also retained as they were not reliably
inferable.

now airport fee

In [0]:
trips_clean.filter(
    F.col("Airport_fee") < 0
).groupBy(
    "Airport_fee"
).count().orderBy(
    "Airport_fee"
).show()

+-----------+-----+
|Airport_fee|count|
+-----------+-----+
|      -1.75| 4921|
+-----------+-----+



In [0]:
trips_clean.filter(
    F.col("Airport_fee") < 0
).select(
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "Airport_fee",
    "total_amount",
    "payment_type"
).show(30)

+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------+------------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|congestion_surcharge|Airport_fee|total_amount|payment_type|
+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------+------------+
|      -19.8| -5.0|   -0.5|       0.0|         0.0|                 -1.0|                 0.0|      -1.75|      -28.05|           4|
|      -22.6| -5.0|   -0.5|       0.0|         0.0|                 -1.0|                 0.0|      -1.75|      -30.85|           4|
|       -3.0|  0.0|   -0.5|       0.0|         0.0|                 -1.0|                 0.0|      -1.75|       -6.25|           3|
|      -70.0|  0.0|   -0.5|       0.0|       -6.94|                 -1.0|                -2.5|      -1.75|      -82.69|           2|
|      -27.5|  0.0|   -0.5|       0.0|         0.0|                 -

### Airport Fee Validation

The `Airport_fee` column contains `-1.75` in 4,921 records.

These records were investigated along with the other monetary columns.
The negative `Airport_fee` values occur with negative `fare_amount`,
`improvement_surcharge`, and `total_amount` values in the same records,
indicating a broader negative-value transaction pattern rather than an
isolated anomaly.

Therefore, the original `Airport_fee` values were retained without
modification. NULL values were also retained as they were not reliably
inferable.

##date time validation

In [0]:
trips_clean.filter(
    F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")
).count()

56

In [0]:
trips_clean.filter(
    F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount",
    "payment_type"
).show(56, truncate=False)

+--------------------+---------------------+-------------+------------+------------+-----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|PULocationID|DOLocationID|fare_amount|total_amount|payment_type|
+--------------------+---------------------+-------------+------------+------------+-----------+------------+------------+
|2024-01-08 12:01:30 |2024-01-08 12:01:18  |1.61         |265         |133         |15.2       |16.0        |0           |
|2024-01-17 23:01:43 |2024-01-17 23:01:05  |4.73         |265         |195         |16.17      |16.97       |0           |
|2024-01-21 04:01:27 |2024-01-21 04:01:04  |7.65         |265         |36          |33.34      |34.14       |0           |
|2024-01-05 11:01:55 |2024-01-05 11:01:33  |2.09         |265         |238         |21.2       |22.0        |0           |
|2024-01-08 21:01:50 |2024-01-08 21:01:26  |6.06         |265         |181         |30.0       |30.0        |0           |
|2024-01-17 10:0

In [0]:
trips_clean.filter(
    F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")
).groupBy(
    "PULocationID"
).count().orderBy(
    F.desc("count")
).show()

+------------+-----+
|PULocationID|count|
+------------+-----+
|         265|   48|
|         192|    1|
|         165|    1|
|          95|    1|
|         133|    1|
|         181|    1|
|          75|    1|
|         179|    1|
|         173|    1|
+------------+-----+



In [0]:
trips_clean.filter(
    F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")
).groupBy(
    "PULocationID",
    "DOLocationID"
).count().orderBy(
    F.desc("count")
).show(20)

+------------+------------+-----+
|PULocationID|DOLocationID|count|
+------------+------------+-----+
|         265|          95|    4|
|         265|         232|    4|
|         265|         235|    2|
|         265|         181|    2|
|         265|          17|    2|
|         265|         133|    2|
|         265|         238|    2|
|         265|          62|    2|
|         181|          89|    1|
|         179|          47|    1|
|         265|         130|    1|
|         265|          36|    1|
|         265|         195|    1|
|         265|         139|    1|
|         133|          89|    1|
|         265|         129|    1|
|         265|          76|    1|
|         265|          25|    1|
|         265|         260|    1|
|          95|          80|    1|
+------------+------------+-----+
only showing top 20 rows


### Datetime Sequence Validation

A total of **56 records** were found where `tpep_dropoff_datetime`
occurs before `tpep_pickup_datetime`.

Further inspection showed that **48 of these 56 records have
`PULocationID = 265`**, while the remaining records are distributed
across other pickup locations.

No reliable evidence was found to determine the intended timestamps.
Therefore, these records were retained with their original datetime
values rather than being arbitrarily modified or removed.

###Trip duration

In [0]:
trips_clean.createOrReplaceTempView("trips")

In [0]:
%sql
SELECT
    MIN(
        TIMESTAMPDIFF(
            SECOND,
            tpep_pickup_datetime,
            tpep_dropoff_datetime
        ) / 60.0
    ) AS min_duration_min,

    MAX(
        TIMESTAMPDIFF(
            SECOND,
            tpep_pickup_datetime,
            tpep_dropoff_datetime
        ) / 60.0
    ) AS max_duration_min,

    AVG(
        TIMESTAMPDIFF(
            SECOND,
            tpep_pickup_datetime,
            tpep_dropoff_datetime
        ) / 60.0
    ) AS avg_duration_min

FROM trips;

min_duration_min,max_duration_min,avg_duration_min
-13.566667,9455.400000,15.6129506190


In [0]:
%sql

SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    TIMESTAMPDIFF(
        SECOND,
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    ) / 60.0 AS trip_duration_min,
    trip_distance,
    PULocationID,
    DOLocationID,
    fare_amount,
    total_amount,
    payment_type
FROM trips
ORDER BY trip_duration_min DESC
LIMIT 20;

tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_min,trip_distance,PULocationID,DOLocationID,fare_amount,total_amount,payment_type
2024-01-24T17:03:14.000,2024-01-31T06:38:38.000,9455.400000,2.26,237,170,30.3,36.8,2
2024-01-12T01:20:50.000,2024-01-14T13:42:36.000,3621.766667,35.7,132,42,150.4,181.41,2
2024-01-14T10:08:11.000,2024-01-16T13:54:22.000,3106.183333,31.95,220,220,2221.3,2225.3,2
2024-01-31T12:39:24.000,2024-02-02T13:56:52.000,2957.466667,0.0,207,260,3.0,4.5,2
2024-01-17T05:31:17.000,2024-01-19T05:12:10.000,2860.883333,0.2,162,162,28.9,33.9,2
2024-01-24T12:39:56.000,2024-01-26T07:45:44.000,2585.800000,0.0,207,226,3.0,4.5,2
2024-01-25T18:53:29.000,2024-01-27T11:32:13.000,2438.733333,4.56,140,133,21.9,28.4,2
2024-01-10T23:28:44.000,2024-01-12T11:02:04.000,2133.333333,37.97,132,132,-70.0,-73.25,4
2024-01-10T23:28:44.000,2024-01-12T11:02:04.000,2133.333333,37.97,132,132,70.0,73.25,4
2024-01-22T06:29:10.000,2024-01-23T17:07:01.000,2077.850000,20.35,132,100,70.0,75.75,2


In [0]:
%sql

SELECT
    CASE
        WHEN trip_duration_min < 0 THEN 'Negative'
        WHEN trip_duration_min <= 10 THEN '0-10 min'
        WHEN trip_duration_min <= 30 THEN '10-30 min'
        WHEN trip_duration_min <= 60 THEN '30-60 min'
        WHEN trip_duration_min <= 120 THEN '60-120 min'
        WHEN trip_duration_min <= 360 THEN '2-6 hours'
        ELSE '>6 hours'
    END AS duration_range,
    COUNT(*) AS trip_count
FROM (
    SELECT
        TIMESTAMPDIFF(
            SECOND,
            tpep_pickup_datetime,
            tpep_dropoff_datetime
        ) / 60.0 AS trip_duration_min
    FROM trips
)
GROUP BY duration_range
ORDER BY trip_count DESC;

duration_range,trip_count
10-30 min,1460649
0-10 min,1233633
30-60 min,239689
60-120 min,27855
>6 hours,1772
2-6 hours,970
Negative,56


In [0]:
%sql

SELECT
    COUNT(*) AS trips_over_6h,
    SUM(CASE WHEN trip_distance = 0 THEN 1 ELSE 0 END) AS zero_distance,
    SUM(CASE WHEN trip_distance > 0 AND trip_distance < 1 THEN 1 ELSE 0 END) AS under_1_mile,
    SUM(CASE WHEN trip_distance >= 1 THEN 1 ELSE 0 END) AS one_or_more_miles
FROM trips
WHERE TIMESTAMPDIFF(
    SECOND,
    tpep_pickup_datetime,
    tpep_dropoff_datetime
) > 6 * 60 * 60;

trips_over_6h,zero_distance,under_1_mile,one_or_more_miles
1772,22,322,1428


### Trip Duration Validation

Trip duration was calculated using the difference between pickup and drop-off timestamps.

* The **average trip duration** is approximately **15.61 minutes**.
* Most trips fall within the **0–30 minute** range.
* **56 records** have a negative duration, where the drop-off timestamp occurs before the pickup timestamp.
* **1,772 records** have a duration greater than **6 hours**.
* Extreme-duration records were inspected to identify potential data quality issues.
* No reliable basis was found to determine the correct duration for these records.

Therefore, the original datetime values were **retained without modification** rather than applying arbitrary corrections or imputations.


In [0]:
trips_clean.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [0]:
trips_clean.count()

2964624

In [0]:
from pyspark.sql import functions as F

trips_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in trips_clean.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         171627|            0|    140162|            140162|           0|           0|           0|          0|    0|      0|         

##Data Enrichment

In [0]:
from pyspark.sql import functions as F

pickup_zones = zones_clean.select(
    F.col("LocationID").alias("PU_LocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
    F.col("service_zone").alias("pickup_service_zone")
)

dropoff_zones = zones_clean.select(
    F.col("LocationID").alias("DO_LocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone"),
    F.col("service_zone").alias("dropoff_service_zone")
)

trips_enriched = trips_clean \
    .join(
        pickup_zones,
        trips_clean.PULocationID == pickup_zones.PU_LocationID,
        "left"
    ) \
    .join(
        dropoff_zones,
        trips_clean.DOLocationID == dropoff_zones.DO_LocationID,
        "left"
    )

In [0]:
trips_enriched.select(
    "PULocationID",
    "pickup_borough",
    "pickup_zone",
    "DOLocationID",
    "dropoff_borough",
    "dropoff_zone"
).show(10, truncate=False)

FILTERING OUT THE DATA ONLY OF JAN 2024

In [0]:
trips_enriched_jan2024 = trips_enriched.filter(
    (F.to_date("tpep_pickup_datetime") >= "2024-01-01") &
    (F.to_date("tpep_pickup_datetime") <= "2024-01-31")
)

In [0]:
print(trips_enriched_jan2024.count())
print(trips_enriched.count())

2964606
2964624


##BUSINESS ANALYSIS

1...Which pickup zones have the highest number of taxi trips?

In [0]:
from pyspark.sql import functions as F

pickup_zone_analysis = trips_enriched_jan2024.groupBy(
    "pickup_borough",
    "pickup_zone"
).agg(
    F.count("*").alias("trip_count")
).orderBy(
    F.desc("trip_count")
)

pickup_zone_analysis.show(10, truncate=False)

+--------------+----------------------------+----------+
|pickup_borough|pickup_zone                 |trip_count|
+--------------+----------------------------+----------+
|Queens        |JFK Airport                 |145240    |
|Manhattan     |Midtown Center              |143469    |
|Manhattan     |Upper East Side South       |142707    |
|Manhattan     |Upper East Side North       |136464    |
|Manhattan     |Midtown East                |106717    |
|Manhattan     |Times Sq/Theatre District   |106324    |
|Manhattan     |Penn Station/Madison Sq West|104522    |
|Manhattan     |Lincoln Square East         |104080    |
|Queens        |LaGuardia Airport           |89530     |
|Manhattan     |Upper West Side South       |88474     |
+--------------+----------------------------+----------+
only showing top 10 rows


2...Busiest Dropoff zone

In [0]:
dropoff_zone_analysis = trips_enriched_jan2024.groupBy(
    "dropoff_borough",
    "dropoff_zone"
).agg(
    F.count("*").alias("trip_count")
).orderBy(
    F.desc("trip_count")
)

dropoff_zone_analysis.show(10, truncate=False)

+---------------+-------------------------+----------+
|dropoff_borough|dropoff_zone             |trip_count|
+---------------+-------------------------+----------+
|Manhattan      |Upper East Side North    |142044    |
|Manhattan      |Upper East Side South    |130247    |
|Manhattan      |Midtown Center           |111942    |
|Manhattan      |Times Sq/Theatre District|90603     |
|Manhattan      |Lincoln Square East      |89672     |
|Manhattan      |Upper West Side South    |89105     |
|Manhattan      |Murray Hill              |86730     |
|Manhattan      |Midtown East             |85238     |
|Manhattan      |Lenox Hill West          |83562     |
|Manhattan      |East Chelsea             |74516     |
+---------------+-------------------------+----------+
only showing top 10 rows


3...Top 10 Routes

In [0]:
route_analysis = trips_enriched_jan2024.groupBy(
    "pickup_zone",
    "dropoff_zone"
).agg(
    F.count("*").alias("trip_count")
).orderBy(
    F.desc("trip_count")
)

route_analysis.show(10, truncate=False)

+---------------------+---------------------+----------+
|pickup_zone          |dropoff_zone         |trip_count|
+---------------------+---------------------+----------+
|Upper East Side South|Upper East Side North|21883     |
|Upper East Side North|Upper East Side South|19402     |
|Upper East Side North|Upper East Side North|15955     |
|Upper East Side South|Upper East Side South|14938     |
|Midtown Center       |Upper East Side South|10275     |
|Lincoln Square East  |Upper West Side South|8980      |
|Upper East Side South|Midtown Center       |8834      |
|Midtown Center       |Upper East Side North|8766      |
|Upper West Side South|Lincoln Square East  |8675      |
|Upper West Side South|Upper West Side North|8445      |
+---------------------+---------------------+----------+
only showing top 10 rows


4...Pickup Zone Performance

In [0]:
pickup_zone_metrics = trips_enriched_jan2024.groupBy(
    "pickup_borough",
    "pickup_zone"
).agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("total_amount"), 2).alias("avg_total")
).orderBy(
    F.desc("trip_count")
)

pickup_zone_metrics.show(15, truncate=False)

+--------------+----------------------------+----------+------------+--------+---------+
|pickup_borough|pickup_zone                 |trip_count|avg_distance|avg_fare|avg_total|
+--------------+----------------------------+----------+------------+--------+---------+
|Queens        |JFK Airport                 |145240    |15.49       |59.4    |76.58    |
|Manhattan     |Midtown Center              |143469    |2.56        |15.21   |23.48    |
|Manhattan     |Upper East Side South       |142707    |1.7         |12.18   |19.45    |
|Manhattan     |Upper East Side North       |136464    |1.85        |12.71   |20.0     |
|Manhattan     |Midtown East                |106717    |2.23        |14.79   |22.88    |
|Manhattan     |Times Sq/Theatre District   |106324    |2.91        |17.54   |26.27    |
|Manhattan     |Penn Station/Madison Sq West|104522    |2.27        |15.79   |23.64    |
|Manhattan     |Lincoln Square East         |104080    |2.09        |13.43   |21.0     |
|Queens        |LaGua

5...Trips by Hour

In [0]:
hourly_trips = trips_enriched_jan2024 \
    .withColumn(
        "pickup_hour",
        F.hour("tpep_pickup_datetime")
    ) \
    .groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("trip_count")
    ) \
    .orderBy("pickup_hour")

hourly_trips.show(24)

+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|          0|     79090|
|          1|     53627|
|          2|     37517|
|          3|     24811|
|          4|     16742|
|          5|     18764|
|          6|     41429|
|          7|     83719|
|          8|    117209|
|          9|    128970|
|         10|    138778|
|         11|    150542|
|         12|    164559|
|         13|    169903|
|         14|    182898|
|         15|    189359|
|         16|    190201|
|         17|    206257|
|         18|    212788|
|         19|    184032|
|         20|    159989|
|         21|    160888|
|         22|    143259|
|         23|    109275|
+-----------+----------+



6...Weekday vs Weekend

In [0]:
daily_trips = trips_enriched_jan2024 \
    .withColumn(
        "day_of_week",
        F.date_format("tpep_pickup_datetime", "EEEE")
    ) \
    .groupBy("day_of_week") \
    .agg(
        F.count("*").alias("trip_count")
    )

daily_trips.show()

+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|     Sunday|    339302|
|   Saturday|    421158|
|     Friday|    408588|
|    Tuesday|    463662|
|     Monday|    408277|
|   Thursday|    428587|
|  Wednesday|    495032|
+-----------+----------+



7...Average Trips per Day of Week

In [0]:
daily_trips_avg = trips_enriched_jan2024 \
    .withColumn(
        "day_of_week",
        F.date_format("tpep_pickup_datetime", "EEEE")
    ) \
    .groupBy("day_of_week") \
    .agg(
        F.count("*").alias("trip_count"),
        F.countDistinct(
            F.to_date("tpep_pickup_datetime")
        ).alias("days")
    ) \
    .withColumn(
        "avg_trips_per_day",
        F.round(F.col("trip_count") / F.col("days"), 0)
    )

daily_trips_avg.orderBy(
    F.desc("avg_trips_per_day")
).show()

+-----------+----------+----+-----------------+
|day_of_week|trip_count|days|avg_trips_per_day|
+-----------+----------+----+-----------------+
|   Thursday|    428587|   4|         107147.0|
|   Saturday|    421158|   4|         105290.0|
|     Friday|    408588|   4|         102147.0|
|  Wednesday|    495032|   5|          99006.0|
|    Tuesday|    463662|   5|          92732.0|
|     Sunday|    339302|   4|          84826.0|
|     Monday|    408277|   5|          81655.0|
+-----------+----------+----+-----------------+



8...Daily revenue trend

In [0]:
daily_revenue = trips_enriched_jan2024 \
    .withColumn(
        "pickup_date",
        F.to_date("tpep_pickup_datetime")
    ) \
    .groupBy("pickup_date") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_value")
    ) \
    .orderBy("pickup_date")

daily_revenue.show(31, truncate=False)

+-----------+----------+-------------+--------------+
|pickup_date|trip_count|total_revenue|avg_trip_value|
+-----------+----------+-------------+--------------+
|2024-01-01 |81013     |2442843.25   |30.15         |
|2024-01-02 |75519     |2282196.02   |30.22         |
|2024-01-03 |82427     |2357588.48   |28.6          |
|2024-01-04 |102901    |2800511.06   |27.22         |
|2024-01-05 |103178    |2728672.52   |26.45         |
|2024-01-06 |97117     |2436272.32   |25.09         |
|2024-01-07 |67543     |1898102.16   |28.1          |
|2024-01-08 |80034     |2216447.23   |27.69         |
|2024-01-09 |93962     |2363882.49   |25.16         |
|2024-01-10 |95000     |2550418.13   |26.85         |
|2024-01-11 |105010    |2905318.19   |27.67         |
|2024-01-12 |103655    |2862882.13   |27.62         |
|2024-01-13 |104758    |2649939.94   |25.3          |
|2024-01-14 |94420     |2459841.63   |26.05         |
|2024-01-15 |77033     |2165047.64   |28.11         |
|2024-01-16 |93057     |2634

9...How does trip volume and average trip value vary across different payment types?

In [0]:
payment_analysis = trips_enriched_jan2024 \
    .groupBy("payment_type") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_value")
    ) \
    .orderBy(F.desc("trip_count"))

payment_analysis.show()

+------------+----------+--------------+
|payment_type|trip_count|avg_trip_value|
+------------+----------+--------------+
|           1|   2319036|         28.26|
|           2|    439185|         22.88|
|           0|    140162|         25.81|
|           4|     46628|          1.77|
|           3|     19595|          8.76|
+------------+----------+--------------+



10...Airport vs Non-Airport Trips

In [0]:
airport_analysis = trips_enriched_jan2024 \
    .withColumn(
        "trip_category",
        F.when(
            F.col("pickup_zone").isin("JFK Airport", "LaGuardia Airport"),
            "Airport"
        ).otherwise("Non-Airport")
    ) \
    .groupBy("trip_category") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_value")
    ) \
    .orderBy(F.desc("trip_count"))

airport_analysis.show()

+-------------+----------+------------+--------------+
|trip_category|trip_count|avg_distance|avg_trip_value|
+-------------+----------+------------+--------------+
|  Non-Airport|   2729836|        2.83|          22.9|
|      Airport|    234770|       13.24|         72.17|
+-------------+----------+------------+--------------+



##optimization

###optimizing broadcasting 

In [0]:
import time
from pyspark.sql import functions as F

start = time.time()

normal_join = trips_clean.join(
    zones_clean,
    trips_clean.PULocationID == zones_clean.LocationID,
    "left"
).select(
    trips_clean.PULocationID,
    zones_clean.Zone
)

normal_join.count()

print(f"Normal Join Time: {time.time() - start:.2f} seconds")

Normal Join Time: 2.52 seconds


In [0]:
import time
from pyspark.sql import functions as F

start = time.time()

broadcast_join = trips_clean.join(
    F.broadcast(zones_clean),
    trips_clean.PULocationID == zones_clean.LocationID,
    "left"
).select(
    trips_clean.PULocationID,
    zones_clean.Zone
)

broadcast_join.count()

print(f"Broadcast Join Time: {time.time() - start:.2f} seconds")

Broadcast Join Time: 2.51 seconds


In [0]:
normal_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (22)
+- == Initial Plan ==
   PhotonResultStage (21)
   +- PhotonColumnarToRow (20)
      +- PhotonProject (19)
         +- PhotonBroadcastHashJoin LeftOuter (18)
            :- PhotonGroupingAgg (6)
            :  +- PhotonShuffleExchangeSource (5)
            :     +- PhotonShuffleMapStage (4)
            :        +- PhotonShuffleExchangeSink (3)
            :           +- PhotonGroupingAgg (2)
            :              +- PhotonScan parquet  (1)
            +- PhotonShuffleExchangeSource (17)
               +- PhotonShuffleMapStage (16)
                  +- PhotonShuffleExchangeSink (15)
                     +- PhotonGroupingAgg (14)
                        +- PhotonShuffleExchangeSource (13)
                           +- PhotonShuffleMapStage (12)
                              +- PhotonShuffleExchangeSink (11)
                                 +- PhotonGroupingAgg (10)
                                    +- PhotonFilter (9)
                    

### Broadcast Join Optimization

A representative join between the `trips_clean` fact table and the
`zones_clean` dimension table was evaluated.

- Normal join execution time: **2.54 seconds**
- Explicit broadcast join execution time: **2.53 seconds**
- Physical plan for the normal join already showed a
  `PhotonBroadcastHashJoin`.

This indicates that Databricks/Photon automatically selected a broadcast
join because `zones_clean` is a small dimension table.

Adding an explicit `broadcast()` hint therefore produced **no meaningful
performance improvement**.

The automatic broadcast configuration could not be disabled in the
Databricks Serverless environment, so an artificial non-broadcast
comparison was not performed.

**Conclusion:** No manual broadcast optimization was required for this
join because Spark/Photon had already selected the appropriate join
strategy automatically.

###optimizing shuffling

In [0]:
import time
from pyspark.sql import functions as F

start = time.time()

shuffle_baseline = trips_enriched_jan2024 \
    .groupBy("pickup_zone") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_value")
    )

shuffle_baseline.count()

print(f"GroupBy Time: {time.time() - start:.2f} seconds")

GroupBy Time: 2.57 seconds


In [0]:
shuffle_baseline.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (40)
+- == Initial Plan ==
   PhotonResultStage (39)
   +- PhotonColumnarToRow (38)
      +- PhotonGroupingAgg (37)
         +- PhotonShuffleExchangeSource (36)
            +- PhotonShuffleMapStage (35)
               +- PhotonShuffleExchangeSink (34)
                  +- PhotonGroupingAgg (33)
                     +- PhotonProject (32)
                        +- PhotonBroadcastHashJoin LeftOuter (31)
                           :- PhotonProject (19)
                           :  +- PhotonBroadcastHashJoin LeftOuter (18)
                           :     :- PhotonGroupingAgg (6)
                           :     :  +- PhotonShuffleExchangeSource (5)
                           :     :     +- PhotonShuffleMapStage (4)
                           :     :        +- PhotonShuffleExchangeSink (3)
                           :     :           +- PhotonGroupingAgg (2)
                           :     :              +- PhotonScan parquet  (1)
                   

In [0]:
import time
from pyspark.sql import functions as F

start = time.time()

shuffle_optimized = trips_enriched_jan2024 \
    .repartition("pickup_zone") \
    .groupBy("pickup_zone") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_value")
    )

shuffle_optimized.count()

print(f"Repartitioned GroupBy Time: {time.time() - start:.2f} seconds")

Repartitioned GroupBy Time: 2.71 seconds


### Shuffle Optimization — Repartitioning

A `groupBy` operation was used as a representative shuffle-intensive workload.

- Baseline groupBy execution time: **2.57 seconds**
- GroupBy after `repartition("pickup_zone")`: **2.71 seconds**
- The repartitioned version was approximately **5.4% slower**.

The physical plan already showed a shuffle for the final grouping operation.
Explicitly repartitioning the data by `pickup_zone` introduced additional
shuffle work before the aggregation and did not improve execution time.

**Conclusion:** Explicit repartitioning was not beneficial for this
one-shot aggregation and was therefore not retained as an optimization.

In [0]:
import time
from pyspark.sql import functions as F

start = time.time()

trips_enriched_jan2024.groupBy("payment_type").count().collect()

trips_enriched_jan2024.groupBy("pickup_zone").count().collect()

print(f"Without Cache Time: {time.time() - start:.2f} seconds")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5886886580662488>, line 6
      2 from pyspark.sql import functions as F
      4 start = time.time()
----> 6 trips_enriched_jan2024.groupBy("payment_type").count().collect()
      8 trips_enriched_jan2024.groupBy("pickup_zone").count().collect()
     10 print(f"Without Cache Time: {time.time() - start:.2f} seconds")

NameError: name 'trips_enriched_jan2024' is not defined

In [0]:
import time
from pyspark.sql import functions as F

trips_cached = trips_enriched_jan2024.cache()

start = time.time()

trips_cached.groupBy("payment_type").count().collect()

trips_cached.groupBy("pickup_zone").count().collect()

print(f"With Cache Time: {time.time() - start:.2f} seconds")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5886886580662489>, line 4
      1 import time
      2 from pyspark.sql import functions as F
----> 4 trips_cached = trips_enriched_jan2024.cache()
      6 start = time.time()
      8 trips_cached.groupBy("payment_type").count().collect()

File <command-8737757436613215>, line 1
----> 1 trips_enriched_jan2024 = trips_enriched.filter(
      2     (F.to_date("tpep_pickup_datetime") >= "2024-01-01") &
      3     (F.to_date("tpep_pickup_datetime") <= "2024-01-31")
      4 )

File <command-7961135600709685>, line 17
      3 pickup_zones = zones_clean.select(
      4     F.col("LocationID").alias("PU_LocationID"),
      5     F.col("Borough").alias("pickup_borough"),
      6     F.col("Zone").alias("pickup_zone"),
      7     F.col("service_zone").alias("pickup_service_zone")
      8 )
     10 dropoff_zones = zones_clean.select(

### Caching / Persistence Limitation

Caching was considered as a Spark optimization for repeated DataFrame
operations.

However, the Databricks Serverless environment does not support
`PERSIST TABLE` / persistence operations required for DataFrame caching.

Therefore, caching could not be benchmarked in this environment and no
artificial workaround was used.

**Conclusion:** Caching was not evaluated as a performance optimization
because it is not supported by the current Serverless compute environment.

In [0]:
trips_enriched_jan2024.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (33)
+- == Initial Plan ==
   PhotonResultStage (32)
   +- PhotonColumnarToRow (31)
      +- PhotonBroadcastHashJoin LeftOuter (30)
         :- PhotonBroadcastHashJoin LeftOuter (18)
         :  :- PhotonGroupingAgg (6)
         :  :  +- PhotonShuffleExchangeSource (5)
         :  :     +- PhotonShuffleMapStage (4)
         :  :        +- PhotonShuffleExchangeSink (3)
         :  :           +- PhotonGroupingAgg (2)
         :  :              +- PhotonScan parquet  (1)
         :  +- PhotonShuffleExchangeSource (17)
         :     +- PhotonShuffleMapStage (16)
         :        +- PhotonShuffleExchangeSink (15)
         :           +- PhotonGroupingAgg (14)
         :              +- PhotonShuffleExchangeSource (13)
         :                 +- PhotonShuffleMapStage (12)
         :                    +- PhotonShuffleExchangeSink (11)
         :                       +- PhotonGroupingAgg (10)
         :                          +- PhotonFilter (9)


### Spark Optimization & Performance Summary

The project included several Spark performance optimization experiments.

- **Broadcast Join:** Databricks/Photon automatically selected a
  `PhotonBroadcastHashJoin` for the small zone dimension table. Explicit
  broadcasting produced no meaningful improvement.
- **Shuffle Optimization:** Explicit `repartition("pickup_zone")` increased
  execution time from **2.57s to 2.71s**, so it was not retained.
- **Caching:** DataFrame caching could not be benchmarked because persistence
  operations are not supported on the current Databricks Serverless compute.
- **AQE:** The physical plan showed `AdaptiveSparkPlan`, confirming that
  Adaptive Query Execution was already active.
- **Photon:** The workload was fully supported by Photon.

Overall, the experiments showed that Spark optimization should be
**evidence-based rather than applied blindly**. The existing Databricks
execution engine already optimized several operations automatically, while
manual repartitioning did not improve performance.